# Notebook 1 — Baseline Evaluation

Measure LFM2.5-1.2B-Thinking accuracy on StepGame **before** fine-tuning.

In [ ]:
# Run once in Colab
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from src.dataset import load_stepgame, format_prompt, extract_answer
from src.eval import evaluate, save_results

In [ ]:
MODEL_ID = 'LiquidAI/LFM2.5-1.2B-Instruct'  # swap to -Thinking if available
EVAL_PATH = '../data/eval/stepgame_eval.json'
OUT_PATH  = '../results/baseline/predictions.json'
MAX_NEW_TOKENS = 128
BATCH_SIZE = 8

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

In [ ]:
examples = load_stepgame(EVAL_PATH)
print(f'Loaded {len(examples)} eval examples')
print('Sample:', examples[0])

In [ ]:
predictions = []

for ex in tqdm(examples):
    prompt = format_prompt(ex['story'], ex['question'])
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    predictions.append({
        'story': ex['story'],
        'question': ex['question'],
        'answer': ex['answer'],
        'prediction': generated,
        'k': ex.get('k'),
    })

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w') as f:
    json.dump(predictions, f, indent=2)

In [ ]:
results = evaluate(predictions)
save_results(results, '../results/baseline/scores.json')

print(f"Overall accuracy: {results['accuracy']:.3f}")
for k, v in results.items():
    if k.startswith('accuracy_k'):
        print(f"  {k}: {v:.3f}")